# Non-overlapping Transformer Forecasts on the Last Test Split

**`SPLIT_MODE = "rows"` (recommended for Excel-style splits):** After sorting by `TIMESTAMP`, use **row indices** — first `TRAIN_RATIO` of rows → train (normalization only), next `VAL_RATIO` → validation band (unused for norm here), **last `TEST_RATIO`** → **test**. Rolling forecasts use **only windows whose input starts in that test row range** (same idea as “last 10% of the sheet”).

**`SPLIT_MODE = "windows"`:** Matches the training pipeline: split **counts of uniform-timestep window starts**, then optionally `ROLL_TEST_TAIL_FRACTION` to plot only the tail of **those** test windows.

Uniform timestep filtering (`REQUIRE_UNIFORM_TIMESTEP`) still applies when stepping the model; **long gaps** on the plot mean some candidates were skipped inside the test rows.


In [7]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from forecast_sweep_common import compute_uniform_timestep_start_indices, parse_timestamp_series
from forecast_sweep_common import row_indices_covered_by_windows, smooth_target_series_1d, split_window_counts
from train_transformer_sweep import TransformerForecastDelta

torch.set_grad_enabled(False)


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [ ]:
# Required: point to the trained .pth checkpoint.
MODEL_PATH = r""

# Optional: point to best_config.json beside the checkpoint.
BEST_CONFIG_PATH = r""

# Defaults from run_transformer_tuning.sbatch.
DATA_CSV = Path("data/AHU_2_9_Blower_DE_A.csv")
TARGET_COLUMN = "Acceleration RMS"
FEATURE_COLUMNS = ["Acceleration RMS"]
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10
TARGET_SMOOTHING_WINDOW = 200
REQUIRE_UNIFORM_TIMESTEP = True
UNIFORM_STEP_SECONDS = 300
UNIFORM_TOL_SECONDS = 30
MIN_WINDOWS_PER_SPLIT = 1

# "rows" = chronological split by sorted row index (80% train / 10% val / 10% test); rolling uses test rows only.
# "windows" = split by counts of uniform window-starts (training-job style).
SPLIT_MODE = "rows"

# Non-overlapping rolling controls.
INPUT_WINDOW_LEN = 432
FORECAST_LEN = 288
WINDOW_STRIDE = 288
MAX_WINDOWS = None
SKIP_NON_UNIFORM_WINDOWS = True

# Only when SPLIT_MODE == "windows": plot the last fraction of test *window-starts*.
ROLL_TEST_TAIL_FRACTION = 1.0

# Plot controls. Lines are split when adjacent points are farther apart than this.
PLOT_GAP_BREAK_SECONDS = 5 * 60

OUTPUT_DIR = Path("rolling_nonoverlap_transformer_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def existing_path_or_none(path_like):
    raw = str(path_like).strip()
    if not raw:
        return None
    path = Path(raw)
    return path if path.exists() else None


cfg_path = existing_path_or_none(BEST_CONFIG_PATH)
run_cfg = {}
if cfg_path is not None:
    run_cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    DATA_CSV = Path(run_cfg.get("single_csv") or DATA_CSV)
    TARGET_COLUMN = run_cfg.get("value_column") or TARGET_COLUMN
    FEATURE_COLUMNS = run_cfg.get("feature_columns") or FEATURE_COLUMNS
    TRAIN_RATIO = float(run_cfg.get("train_ratio_window") or TRAIN_RATIO)
    VAL_RATIO = float(run_cfg.get("val_ratio_window") or VAL_RATIO)
    TEST_RATIO = float(run_cfg.get("test_ratio_window") or TEST_RATIO)
    TARGET_SMOOTHING_WINDOW = int(run_cfg.get("target_smoothing_window") or TARGET_SMOOTHING_WINDOW)
    REQUIRE_UNIFORM_TIMESTEP = bool(run_cfg.get("require_uniform_timestep", REQUIRE_UNIFORM_TIMESTEP))
    UNIFORM_STEP_SECONDS = float(run_cfg.get("uniform_step_seconds") or UNIFORM_STEP_SECONDS)
    UNIFORM_TOL_SECONDS = float(run_cfg.get("uniform_step_tolerance_seconds") or UNIFORM_TOL_SECONDS)
    MIN_WINDOWS_PER_SPLIT = int(run_cfg.get("min_windows_per_split") or MIN_WINDOWS_PER_SPLIT)

model_path = existing_path_or_none(MODEL_PATH)
if model_path is None or not model_path.is_file():
    raise FileNotFoundError("Set MODEL_PATH to your trained .pth checkpoint before running the notebook.")
if not DATA_CSV.exists():
    raise FileNotFoundError(f"DATA_CSV not found: {DATA_CSV}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
model_cfg = dict(checkpoint.get("model_config") or run_cfg.get("model_config") or {})
ckpt_input_len = int(checkpoint.get("input_len") or model_cfg.get("input_len"))
ckpt_pred_len = int(checkpoint.get("pred_len") or model_cfg.get("pred_len"))
input_dim = int(model_cfg.get("input_dim", len(FEATURE_COLUMNS)))

if INPUT_WINDOW_LEN > ckpt_input_len:
    raise ValueError(f"INPUT_WINDOW_LEN={INPUT_WINDOW_LEN} > checkpoint input_len={ckpt_input_len}")
if FORECAST_LEN > ckpt_pred_len:
    raise ValueError(f"FORECAST_LEN={FORECAST_LEN} > checkpoint pred_len={ckpt_pred_len}")
if input_dim != len(FEATURE_COLUMNS):
    raise ValueError(f"Checkpoint input_dim={input_dim}, but FEATURE_COLUMNS has {len(FEATURE_COLUMNS)} columns")

model = TransformerForecastDelta(
    seq_len=ckpt_input_len,
    input_dim=input_dim,
    pred_len=ckpt_pred_len,
    d_model=int(model_cfg.get("d_model", 128)),
    nhead=int(model_cfg.get("nhead", 8)),
    num_layers=int(model_cfg.get("num_layers", 4)),
    dim_feedforward=int(model_cfg.get("dim_feedforward", 256)),
    dropout=float(model_cfg.get("dropout", 0.1)),
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded model on {device}: checkpoint input_len={ckpt_input_len}, pred_len={ckpt_pred_len}")


In [ ]:
df = pd.read_csv(DATA_CSV)
df["TIMESTAMP"] = parse_timestamp_series(df["TIMESTAMP"], str(DATA_CSV))
df = df.sort_values("TIMESTAMP").reset_index(drop=True)

missing_cols = [c for c in [TARGET_COLUMN, *FEATURE_COLUMNS] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in {DATA_CSV}: {missing_cols}")

smooth_window = TARGET_SMOOTHING_WINDOW if TARGET_SMOOTHING_WINDOW % 2 == 1 else TARGET_SMOOTHING_WINDOW + 1
if smooth_window > 1:
    df[TARGET_COLUMN] = smooth_target_series_1d(df[TARGET_COLUMN].to_numpy(dtype=np.float32), smooth_window)

target = df[TARGET_COLUMN].to_numpy(dtype=np.float32)
features = df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
split_span = ckpt_input_len + ckpt_pred_len
if REQUIRE_UNIFORM_TIMESTEP:
    valid_starts = compute_uniform_timestep_start_indices(df["TIMESTAMP"], split_span, UNIFORM_STEP_SECONDS, UNIFORM_TOL_SECONDS)
else:
    valid_starts = np.arange(max(0, len(df) - split_span + 1), dtype=np.int64)

n = len(df)
mode = str(SPLIT_MODE).strip().lower()
last_test_window_start = None

if mode == "rows":
    rsum = float(TRAIN_RATIO) + float(VAL_RATIO) + float(TEST_RATIO)
    if abs(rsum - 1.0) > 1e-2:
        raise ValueError(f"SPLIT_MODE='rows': TRAIN_RATIO+VAL_RATIO+TEST_RATIO should sum to 1.0, got {rsum}")
    train_end = int(n * float(TRAIN_RATIO))
    val_end = train_end + int(n * float(VAL_RATIO))
    val_end = min(val_end, n)
    test_begin = val_end
    if test_begin >= n:
        raise ValueError("Row split: empty test region (increase data length or adjust ratios).")
    train_rows = np.arange(0, train_end, dtype=np.int64)
    if train_rows.size == 0:
        raise ValueError("Row split: no train rows.")
    plot_roll_start_row = int(test_begin)
    first_test_start_row = plot_roll_start_row
    n_train = n_val = n_test = None
elif mode == "windows":
    n_train, n_val, n_test = split_window_counts(int(valid_starts.size), TRAIN_RATIO, VAL_RATIO, TEST_RATIO, MIN_WINDOWS_PER_SPLIT)
    train_starts = valid_starts[:n_train]
    test_starts = valid_starts[n_train + n_val:]
    if test_starts.size == 0:
        raise ValueError("No test starts were reconstructed.")
    train_rows = row_indices_covered_by_windows(train_starts, split_span, len(df))
    first_test_start_row = int(test_starts[0])
    frac = float(np.clip(ROLL_TEST_TAIL_FRACTION, 1e-9, 1.0))
    n_tail = max(1, int(np.ceil(n_test * frac)))
    tail_test_starts = test_starts[-n_tail:]
    plot_roll_start_row = int(tail_test_starts[0])
    last_test_window_start = int(test_starts[-1])
else:
    raise ValueError(f"SPLIT_MODE must be 'rows' or 'windows', got {SPLIT_MODE!r}")

train_mean = float(checkpoint.get("train_mean", np.mean(target[train_rows])))
train_std = float(checkpoint.get("train_std", np.std(target[train_rows]) + 1e-8))
feat_mean = features[train_rows].mean(axis=0)
feat_std = features[train_rows].std(axis=0) + 1e-8

roll_span = int(INPUT_WINDOW_LEN + FORECAST_LEN)
if roll_span != split_span:
    print(
        f"Note: ckpt split_span={split_span} vs INPUT_WINDOW_LEN+FORECAST_LEN={roll_span}."
    )

if mode == "rows":
    print(
        f"SPLIT_MODE=rows: train rows [0, {train_end}), val [{train_end}, {val_end}), "
        f"test [{test_begin}, {n}) — normalization uses train rows only ({train_rows.size} rows)."
    )
    print(
        f"Rolling forecasts use window starts in [{plot_roll_start_row}, …] up to data end "
        f"(first input row {plot_roll_start_row}: {df['TIMESTAMP'].iloc[plot_roll_start_row]})."
    )
elif mode == "windows":
    print(f"Valid starts: total={len(valid_starts)}, train={n_train}, val={n_val}, test={n_test}")
    print(
        f"Rolling plot uses last {n_tail}/{n_test} test starts (ROLL_TEST_TAIL_FRACTION={frac:g}); "
        f"first rolling input row {plot_roll_start_row} at {df['TIMESTAMP'].iloc[plot_roll_start_row]}"
    )
    print(f"First test window starts at row {first_test_start_row}: {df['TIMESTAMP'].iloc[first_test_start_row]}")
    last_test_start = int(test_starts[-1])
    last_test_end_row = last_test_start + split_span - 1
    print(
        f"Last test window (split): rows {last_test_start}→{last_test_end_row}, "
        f"time {df['TIMESTAMP'].iloc[last_test_start]} → {df['TIMESTAMP'].iloc[last_test_end_row]}"
    )

print(f"Uniform span for valid_starts diag: split_span={split_span} @ {UNIFORM_STEP_SECONDS}s ± {UNIFORM_TOL_SECONDS}s")
print(f"Full CSV after sort: rows 0..{n - 1}, time {df['TIMESTAMP'].iloc[0]} → {df['TIMESTAMP'].iloc[-1]}")
last_any_uniform_start = int(valid_starts[-1]) if len(valid_starts) else -1
if last_any_uniform_start >= 0:
    last_any_uniform_end = last_any_uniform_start + split_span - 1
    if last_any_uniform_end < n - 1:
        print(
            f"Trailing rows after last uniform window end ({df['TIMESTAMP'].iloc[last_any_uniform_end]}): "
            f"{n - 1 - last_any_uniform_end} row(s)."
        )


In [ ]:
def window_is_uniform(start, length):
    if not SKIP_NON_UNIFORM_WINDOWS or length <= 1:
        return True
    ts = df["TIMESTAMP"].iloc[start:start + length].to_numpy(dtype="datetime64[ns]").astype("int64")
    diffs = np.diff(ts) / 1e9
    return bool(np.all(np.abs(diffs - UNIFORM_STEP_SECONDS) <= UNIFORM_TOL_SECONDS))


def predict_from_ground_truth_window(start):
    x_raw = features[start:start + INPUT_WINDOW_LEN]
    x_norm = (x_raw - feat_mean) / feat_std
    x_tensor = torch.tensor(x_norm.T[None, :, :], dtype=torch.float32, device=device)
    last_val_norm = (target[start + INPUT_WINDOW_LEN - 1] - train_mean) / train_std
    pred_delta = model(x_tensor).detach().cpu().numpy()[0]
    pred_abs_norm = pred_delta + float(last_val_norm)
    return (pred_abs_norm * train_std + train_mean)[:FORECAST_LEN]


rows = []
n_skip_uniform = 0
per_window_dir = OUTPUT_DIR / "per_window_predictions"
per_window_dir.mkdir(parents=True, exist_ok=True)
max_start = len(df) - INPUT_WINDOW_LEN - FORECAST_LEN
_mode = str(SPLIT_MODE).strip().lower()
if _mode == "rows":
    upper_start = int(max_start)
elif _mode == "windows":
    upper_start = int(min(max_start, last_test_window_start))
else:
    raise ValueError(f"SPLIT_MODE must be 'rows' or 'windows', got {SPLIT_MODE!r}")

if plot_roll_start_row > upper_start:
    raise ValueError(
        f"No rolling candidates: plot_roll_start_row={plot_roll_start_row} > upper_start={upper_start}. "
        "Test region may be shorter than one input+forecast span; shorten INPUT_WINDOW_LEN/FORECAST_LEN or use more rows."
    )
candidate_starts = np.arange(plot_roll_start_row, upper_start + 1, WINDOW_STRIDE, dtype=np.int64)
if MAX_WINDOWS is not None:
    candidate_starts = candidate_starts[:int(MAX_WINDOWS)]

for source_idx, start in enumerate(candidate_starts):
    if not window_is_uniform(int(start), INPUT_WINDOW_LEN + FORECAST_LEN):
        n_skip_uniform += 1
        print(f"Skipping source window {source_idx} at row {start}: non-uniform timestamps")
        continue
    pred = predict_from_ground_truth_window(int(start))
    forecast_start = int(start) + INPUT_WINDOW_LEN
    forecast_rows = np.arange(forecast_start, forecast_start + FORECAST_LEN, dtype=np.int64)
    window_df = pd.DataFrame({
        "source_window_index": source_idx,
        "source_input_start_row": int(start),
        "forecast_window_index": source_idx + 1,
        "row_index": forecast_rows,
        "step_ahead": np.arange(1, FORECAST_LEN + 1),
        "timestamp": df["TIMESTAMP"].iloc[forecast_rows].astype(str).to_list(),
        "predicted": pred,
        "actual": target[forecast_rows],
    })
    window_df.to_csv(per_window_dir / f"prediction_window_{source_idx + 1:04d}.csv", index=False)
    rows.append(window_df)

if not rows:
    raise ValueError("No rolling prediction windows were produced.")
combined = pd.concat(rows, ignore_index=True)
combined_path = OUTPUT_DIR / "combined_nonoverlap_predictions.csv"
combined.to_csv(combined_path, index=False)

input_rows = np.arange(plot_roll_start_row, plot_roll_start_row + INPUT_WINDOW_LEN, dtype=np.int64)
first_input = pd.DataFrame({
    "row_index": input_rows,
    "timestamp": df["TIMESTAMP"].iloc[input_rows].astype(str).to_list(),
    "ground_truth_input": target[input_rows],
})
first_input_path = OUTPUT_DIR / "first_input_ground_truth_window.csv"
first_input.to_csv(first_input_path, index=False)
ts_plot_min = min(
    pd.to_datetime(first_input["timestamp"]).min(),
    pd.to_datetime(combined["timestamp"]).min(),
)
ts_plot_max = max(
    pd.to_datetime(first_input["timestamp"]).max(),
    pd.to_datetime(combined["timestamp"]).max(),
)
print(
    f"Rolling plot timestamp range: {ts_plot_min} → {ts_plot_max} "
    f"(vs CSV {df['TIMESTAMP'].iloc[0]} → {df['TIMESTAMP'].iloc[-1]})"
)
print(
    f"Candidates: {len(candidate_starts)}, windows saved: {len(rows)}, "
    f"skipped (non-uniform over roll span): {n_skip_uniform}. "
    "Large skips → disjoint blobs on the x-axis; that is expected."
)
print(f"Saved {len(rows)} prediction windows to {per_window_dir}")


In [ ]:
plot_input = first_input.copy()
plot_pred = combined.copy()
plot_input["timestamp"] = pd.to_datetime(plot_input["timestamp"])
plot_pred["timestamp"] = pd.to_datetime(plot_pred["timestamp"])


def plot_with_gap_breaks(ax, frame, x_col, y_col, *, max_gap_seconds, label, **plot_kwargs):
    frame = frame.sort_values(x_col).reset_index(drop=True)
    if frame.empty:
        return

    gaps = frame[x_col].diff().dt.total_seconds().gt(max_gap_seconds).fillna(False)
    segment_ids = gaps.cumsum()
    first_segment = True
    for _, segment in frame.groupby(segment_ids, sort=False):
        ax.plot(
            segment[x_col],
            segment[y_col],
            label=label if first_segment else None,
            **plot_kwargs,
        )
        first_segment = False


fig, ax = plt.subplots(figsize=(18, 5))
plot_with_gap_breaks(
    ax,
    plot_input,
    "timestamp",
    "ground_truth_input",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
    color="0.25",
    linewidth=1.2,
    label=f"First ground-truth input window ({INPUT_WINDOW_LEN})",
)
plot_with_gap_breaks(
    ax,
    plot_pred,
    "timestamp",
    "actual",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
    color="C0",
    linewidth=1.0,
    label=f"Ground truth over forecast windows ({FORECAST_LEN} each)",
)
plot_with_gap_breaks(
    ax,
    plot_pred,
    "timestamp",
    "predicted",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
    color="C1",
    linewidth=1.0,
    label=f"Predicted windows from window 2 onward ({FORECAST_LEN} each)",
)
ax.axvline(plot_input["timestamp"].iloc[-1], color="0.55", linestyle="--", linewidth=1.0)
ax.set_title("First test input window + ground truth vs predicted non-overlapping windows")
ax.set_xlabel("Timestamp")
ax.set_ylabel(TARGET_COLUMN)
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()
plot_path = OUTPUT_DIR / "first_input_then_ground_truth_vs_predictions.png"
fig.savefig(plot_path, dpi=150)
print(f"Saved plot to {plot_path}")


In [ ]:
# Interactive zoomable version of the same plot.
# If this import fails, install Plotly in the notebook kernel: pip install plotly
import plotly.graph_objects as go


def trace_arrays_with_gap_breaks(frame, x_col, y_col, *, max_gap_seconds):
    frame = frame.sort_values(x_col).reset_index(drop=True)
    xs = []
    ys = []
    for idx, row in frame.iterrows():
        if idx > 0:
            gap_seconds = (row[x_col] - frame.loc[idx - 1, x_col]).total_seconds()
            if gap_seconds > max_gap_seconds:
                xs.append(None)
                ys.append(None)
        xs.append(row[x_col])
        ys.append(row[y_col])
    return xs, ys


x_input, y_input = trace_arrays_with_gap_breaks(
    plot_input,
    "timestamp",
    "ground_truth_input",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
)
x_actual, y_actual = trace_arrays_with_gap_breaks(
    plot_pred,
    "timestamp",
    "actual",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
)
x_pred, y_pred = trace_arrays_with_gap_breaks(
    plot_pred,
    "timestamp",
    "predicted",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_input,
        y=y_input,
        mode="lines",
        name=f"First ground-truth input window ({INPUT_WINDOW_LEN})",
        line=dict(color="rgba(64,64,64,1)", width=1.6),
    )
)
fig.add_trace(
    go.Scatter(
        x=x_actual,
        y=y_actual,
        mode="lines",
        name=f"Ground truth over forecast windows ({FORECAST_LEN} each)",
        line=dict(color="#1f77b4", width=1.3),
    )
)
fig.add_trace(
    go.Scatter(
        x=x_pred,
        y=y_pred,
        mode="lines",
        name=f"Predicted windows from window 2 onward ({FORECAST_LEN} each)",
        line=dict(color="#ff7f0e", width=1.3),
    )
)
fig.add_vline(
    x=plot_input["timestamp"].iloc[-1],
    line_dash="dash",
    line_color="gray",
    line_width=1,
)
fig.update_layout(
    title="Interactive: first test input window + ground truth vs predicted non-overlapping windows",
    xaxis_title="Timestamp",
    yaxis_title=TARGET_COLUMN,
    hovermode="x unified",
    template="plotly_white",
    width=1200,
    height=520,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_xaxes(rangeslider_visible=True)

interactive_plot_path = OUTPUT_DIR / "interactive_ground_truth_vs_predictions.html"
fig.write_html(interactive_plot_path)
print(f"Saved interactive plot to {interactive_plot_path}")
fig.show()
